# MIRAGE 재현(Colab): 환경 설정 → 학습 → 생성 → 시각화

저장소: https://github.com/MedARC-AI/MIRAGE (README 기준). 이 노트북은 README에 있는 사실만 근거로 만들었고, **GPU에서 실제로 실행해 검증하지는 못했습니다**(로컬에 GPU가 없음). 각 단계 끝의 점검 셀(✅/⚠️)이 어디서 막히는지 알려 줍니다.

## 먼저 알아둘 것
- **GPU:** README에는 VRAM 요구량이 없습니다. Stable Cascade 단계 C(3.6B, bf16 약 7.2GB)와 단계 B(3B, bf16 약 3.1GB)를 코드가 오프로딩 없이 GPU에 올립니다. 그래서 **A100(40GB 이상) 권장**, L4(24GB)는 빠듯할 수 있다는 것은 제 추정입니다.
- **RAM:** 학습 노트북이 학습 이미지를 float32로 메모리에 올립니다. **고용량 RAM 런타임**을 쓰세요.
- **디스크:** `coco_images_224_float16.hdf5`만 22GB입니다. 체크포인트 약 11GB, 피험자 1 베타 약 1.9GB가 더해집니다. 전체 README의 300GB 다운로드는 필요 없고, 이 노트북은 필요한 파일만 받습니다.
- **Python 버전:** MIRAGE는 torch 2.1.0 등을 고정합니다. 현재 Colab의 기본 Python에는 이 버전이 없을 수 있어서, `uv`로 **Python 3.10 가상환경**을 따로 만들어 그 안에서 MIRAGE 노트북을 실행합니다.
- **생성 시간 줄이기:** 기본값(gen_rep=10, num_images_per_sample=16)은 자극 하나당 생성이 160번입니다. 첫 실행은 아래 설정 셀의 값을 작게 두세요.

## 확인하지 못한 것 (실행 중 막히면 여기부터)
1. MIRAGE용 릿지 가중치(`ridge_*_weights.pkl`)를 미리 제공하는지: 공개 목록에서 찾지 못해 이 노트북은 **직접 학습**합니다.
2. 임베딩 캐시 `git_image_features.hdf5`(73k장 × 257 × 1024)는 매우 큽니다. 기본값은 프롬프트(캡션 생성) 구성요소를 끄고 건너뜁니다.
3. `imagery_stimuli_18.pt`(18장, 3×425×425)를 만드는 코드는 저장소 파일에서 찾지 못했습니다. 아래에서 NSD-Imagery 원본 자극으로 만들어 보는 셀이 있으며 **자극 순서 가정**을 반드시 확인해야 합니다.
4. NSD S3 버킷의 NSD-Imagery 자극 파일 위치.


In [ ]:
# 1. GPU / 메모리 / 디스크 점검
!nvidia-smi
!free -h | head -2
!df -h /content | tail -1

In [ ]:
# 2. 설정 (여기만 바꾸면 됩니다)
import os
ROOT        = "/content/MIRAGE"           # 저장소가 놓일 위치
SUBJ        = 1                           # 피험자 (1~8)
NUM_SESSIONS= 40                          # 학습 세션 수. 빠른 점검은 10, 논문 설정은 40
MODEL_NAME  = f"subj0{SUBJ}_{NUM_SESSIONS}sess_colab_ridge_sc2"
CAPTION_TYPE= "medium"                    # coco / short / medium / schmedium
USE_PROMPT_RECON = False                  # True면 GIT 캡션 프롬프트 사용(73k장 특징 계산: 매우 무겁다)
USE_BLURRY  = True                        # VDVAE 저수준 재구성
USE_DUAL_GUIDANCE = True
MODES       = ["imagery"]                 # "vision", "imagery", "shared1000"
GEN_REP     = 1                           # 기본 10. 첫 실행은 1
N_IMG       = 4                           # num_images_per_sample, 기본 16. 첫 실행은 4
NUM_TRIAL_REPS = 16                       # 기본 16
DRIVE_DIR   = "/content/drive/MyDrive/mirage_colab"  # 결과 백업(선택)
HF_TOKEN    = None                        # 필요하면 문자열로 지정

DATA   = f"{ROOT}/dataset"                # README의 ../dataset
CACHE  = f"{ROOT}/cache"                  # README의 ../cache
SRC    = f"{ROOT}/src"
VENV   = "/content/mei-env"
for d in (ROOT, DATA, f"{DATA}/preprocessed_data/subject{SUBJ}", CACHE):
    os.makedirs(d, exist_ok=True)
print(MODEL_NAME)

In [ ]:
# 3. (선택) Google Drive 마운트: 체크포인트와 결과를 끊김에서 지키기 위한 백업용
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)

## A. 환경 설정 (Python 3.10 가상환경)
`setup.sh`의 고정 버전을 그대로 쓰되, `venv` 대신 `uv`로 Python 3.10을 준비합니다.

In [ ]:
%%bash
set -e
pip -q install uv
uv python install 3.10
uv venv /content/mei-env --python 3.10
echo "venv ok"


In [ ]:
%%bash
set -e
PY=/content/mei-env/bin/python
# 핵심 패키지(setup.sh의 고정 버전). 하나라도 실패하면 여기서 멈춘다.
uv pip install --python $PY \
  numpy==1.26.3 matplotlib==3.8.2 tqdm scikit-image==0.22.0 accelerate==0.24.1 webdataset==0.2.73 \
  pandas==2.2.0 einops==0.7.0 ftfy regex kornia==0.7.1 h5py==3.10.0 open_clip_torch==2.24.0 \
  torch==2.1.0 torchvision==0.16.0 transformers==4.37.2 xformers==0.0.22.post7 torchmetrics==1.3.0.post0 \
  diffusers==0.30.3 omegaconf==2.3.0 pytorch-lightning==2.0.1 sentence-transformers==2.5.1 \
  nibabel==5.2.1 munch==4.0.0 safetensors==0.4.2 huggingface-hub==0.20.3 timm==0.9.12 scipy==1.12.0 \
  pillow==10.2.0 ipykernel jupyter nbconvert nbformat


In [ ]:
%%bash
PY=/content/mei-env/bin/python
# setup.sh의 나머지(선택적/무거운 것들은 실패해도 계속). 필요한 import가 없다고 나오면 그때 다시 시도하세요.
uv pip install --python $PY git+https://github.com/ildoonet/pytorch-gradual-warmup-lr.git || echo "WARN gradual-warmup"
uv pip install --python $PY git+https://github.com/pabloppp/pytorch-tools || echo "WARN pytorch-tools"
uv pip install --python $PY git+https://github.com/openai/CLIP.git --no-deps || echo "WARN CLIP"
uv pip install --python $PY dalle2-pytorch || echo "WARN dalle2-pytorch"
uv pip install --python $PY wandb deepspeed==0.13.1 evaluate==0.4.1 nltk==3.8.1 rouge_score==0.1.2 || echo "WARN optional"
uv pip install --python $PY insightface==0.7.3 opencv-python==4.8.1.78 onnxruntime==1.16.3 onnx2torch==1.5.13 || echo "WARN insightface/onnx (추론에는 불필요할 가능성이 큼, 확인 안 함)"
$PY -m ipykernel install --user --name mei-env --display-name "mei-env (py3.10)"
$PY -c "import torch, transformers, diffusers; print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); print('transformers', transformers.__version__, 'diffusers', diffusers.__version__)"


In [ ]:
# 4. 저장소와 StableCascade 클론 (README: src/StableCascade/ 위치)
!git clone -q https://github.com/MedARC-AI/MIRAGE.git {ROOT}_repo || echo "이미 있음"
!rsync -a {ROOT}_repo/ {ROOT}/
!git clone -q https://github.com/Stability-AI/StableCascade.git {SRC}/StableCascade || echo "이미 있음"
!ls {SRC} | head -30

## B. 체크포인트와 데이터 받기 (필요한 파일만)
- 체크포인트(`reesekneeland/MIRAGE`): 단계 A/B/C, effnet, previewer, VDVAE.
- 캡션(`reesekneeland/MIRAGE`): 학습용 캡션 .npy.
- NSD 베타·COCO 이미지(`pscotti/mindeyev2`): 피험자 베타 약 1.9GB, COCO 224 이미지 22GB.


In [ ]:
# 5. Hugging Face에서 필요한 파일만 받기
%pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
import shutil
def get(repo, fn, dst_dir, subdir=None):
    p = hf_hub_download(repo_id=repo, filename=fn, repo_type="dataset", local_dir=dst_dir, token=HF_TOKEN)
    print("ok", fn)
    return p
ckpt = ["effnet_encoder.safetensors", "previewer.safetensors", "stage_a.safetensors",
        "stage_b_bf16.safetensors", "stage_c_bf16.safetensors", "imagenet64-iter-1600000-model-ema.th"]
for fn in ckpt:
    get("reesekneeland/MIRAGE", fn, CACHE)
caps = {"coco": "annots_73k.npy", "short": "short_length_captions.npy", "medium": "mid_length_captions_73K.npy"}
for k in ("coco", CAPTION_TYPE):
    if k in caps:
        get("reesekneeland/MIRAGE", caps[k], f"{DATA}/preprocessed_data")
get("pscotti/mindeyev2", f"betas_all_subj0{SUBJ}_fp32_renorm.hdf5", DATA)
# 22GB: 시간이 오래 걸린다
get("pscotti/mindeyev2", "coco_images_224_float16.hdf5", DATA)
get("pscotti/mindeyev2", "shared1000.npy", DATA)

In [ ]:
# 6. NSD 쪽 파일(공개 S3 버킷, 인증 불필요): 자극 정보 CSV, 피험자 imagery 베타, nsdgeneral 마스크
%pip -q install awscli
import subprocess, textwrap
S3 = "s3://natural-scenes-dataset"
def s3cp(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    r = subprocess.run(["aws", "s3", "cp", "--no-sign-request", f"{S3}/{src}", dst], capture_output=True, text=True)
    print(("ok  " if r.returncode == 0 else "FAIL"), src, (r.stderr[-200:] if r.returncode else ""))
s = f"subj0{SUBJ}"
s3cp("nsddata/experiments/nsd/nsd_stim_info_merged.csv", f"{DATA}/nsddata/experiments/nsd/nsd_stim_info_merged.csv")
s3cp(f"nsddata/ppdata/{s}/func1pt8mm/roi/nsdgeneral.nii.gz", f"{DATA}/nsddata/ppdata/{s}/func1pt8mm/roi/nsdgeneral.nii.gz")
s3cp(f"nsddata_betas/ppdata/{s}/func1pt8mm/nsdimagerybetas_fithrf_GLMdenoise_RR/betas_nsdimagery.nii.gz",
     f"{DATA}/nsddata_betas/ppdata/{s}/func1pt8mm/nsdimagerybetas_fithrf_GLMdenoise_RR/betas_nsdimagery.nii.gz")
# 자극 파일 위치를 모르므로 버킷에서 찾아본다
r = subprocess.run(["aws", "s3", "ls", "--no-sign-request", f"{S3}/nsddata_stimuli/stimuli/", "--recursive"], capture_output=True, text=True)
print("\n".join([l for l in r.stdout.splitlines() if "imagery" in l.lower()][:30]) or "imagery 관련 항목을 못 찾음: 아래 7번의 업로드 대안을 쓰세요")

In [ ]:
# 7. 위에서 찾은 NSD-Imagery 자극 파일을 받아 로더가 기대하는 두 위치에 둔다.
# 위 출력에 나온 경로로 아래 KEY를 고치세요(예: nsddata_stimuli/stimuli/nsd/nsdimagery_stimuli.pkl3).
KEY = "nsddata_stimuli/stimuli/nsd/nsdimagery_stimuli.pkl3"
for dst in (f"{DATA}/nsddata_stimuli/stimuli/nsdimagery_stimuli.pkl3", f"{DATA}/nsddata_stimuli/stimuli/nsd/nsdimagery_stimuli.pkl3"):
    s3cp(KEY, dst)
# S3에 없으면: 직접 내려받은 파일을 Colab에 업로드해 위 두 경로에 복사하세요.
# imagery_stimuli_18.pt는 만드는 코드를 저장소에서 찾지 못했습니다(형태 (18,3,425,425)). 아래 8번에서 만들어 봅니다.

In [ ]:
# 8. imagery_stimuli_18.pt 만들기(가정 포함, 반드시 확인): 18장 = 세트 A 6 + 세트 B 6 + 세트 C 6 순서라고 가정.
# 세트 A·B 이미지는 NSD-Imagery 원본 자극 폴더(rawtargetimages/setA, setB)를 업로드하고, 세트 C(단어)는 이미지가 없을 수 있다.
# 가진 이미지만으로 채우고(나머지 0), 자극 순서는 pkl3의 image_map·cues와 대조해서 고치세요.
import pickle, torch, numpy as np
from PIL import Image
P = f"{DATA}/nsddata_stimuli/stimuli/nsdimagery_stimuli.pkl3"
if os.path.exists(P):
    d = pickle.load(open(P, "rb"))
    print("pkl3 keys:", list(d.keys()) if hasattr(d, "keys") else type(d))
    for k in ("exps", "cues", "image_map"):
        v = d.get(k) if hasattr(d, "get") else None
        print(k, type(v), (len(v) if hasattr(v, "__len__") else ""))
    print(str(d.get("image_map"))[:800] if hasattr(d, "get") else "")
else:
    print("pkl3가 없음: 7번을 먼저 해결하세요")
RAW = "/content/rawtargetimages"   # 업로드한 setA, setB 폴더 위치
def load_set(folder, n=6):
    fs = sorted(os.listdir(folder))[:n] if os.path.isdir(folder) else []
    ims = [np.asarray(Image.open(f"{folder}/{f}").convert("RGB").resize((425, 425)), dtype=np.float32) / 255.0 for f in fs]
    ims += [np.zeros((425, 425, 3), np.float32)] * (n - len(ims))
    return ims
imgs = load_set(f"{RAW}/setA") + load_set(f"{RAW}/setB") + [np.zeros((425, 425, 3), np.float32)] * 6
T = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).contiguous()
print(T.shape)
os.makedirs(f"{DATA}/nsddata_stimuli/stimuli", exist_ok=True)
torch.save(T, f"{DATA}/nsddata_stimuli/stimuli/imagery_stimuli_18.pt")

In [ ]:
# 9. 사전 점검: 필요한 파일이 모두 있는지
need = [f"{CACHE}/{f}" for f in ["effnet_encoder.safetensors", "previewer.safetensors", "stage_a.safetensors", "stage_b_bf16.safetensors", "stage_c_bf16.safetensors", "imagenet64-iter-1600000-model-ema.th"]]
need += [f"{DATA}/betas_all_subj0{SUBJ}_fp32_renorm.hdf5", f"{DATA}/coco_images_224_float16.hdf5", f"{DATA}/preprocessed_data/annots_73k.npy",
         f"{DATA}/nsddata/experiments/nsd/nsd_stim_info_merged.csv", f"{DATA}/nsddata/ppdata/subj0{SUBJ}/func1pt8mm/roi/nsdgeneral.nii.gz",
         f"{DATA}/nsddata_betas/ppdata/subj0{SUBJ}/func1pt8mm/nsdimagerybetas_fithrf_GLMdenoise_RR/betas_nsdimagery.nii.gz",
         f"{DATA}/nsddata_stimuli/stimuli/nsdimagery_stimuli.pkl3", f"{DATA}/nsddata_stimuli/stimuli/imagery_stimuli_18.pt"]
if CAPTION_TYPE in caps:
    need.append(f"{DATA}/preprocessed_data/{caps[CAPTION_TYPE]}")
missing = [p for p in need if not os.path.exists(p)]
print("✅ 모두 있음" if not missing else "⚠️ 없음:\n" + "\n".join(missing))

## C. 학습 (릿지 회귀, perception 데이터만)
저장소의 `src/Train.ipynb`를 **인자만 바꿔서** 3.10 가상환경 커널로 실행합니다. 결과는 `train_logs/{MODEL_NAME}/ridge_*_weights.pkl`입니다.
- `USE_PROMPT_RECON=False`이면 GIT 캡션 프롬프트 구성요소를 끕니다(73k장 특징 계산 회피). 이 경우 노트북이 `git_image_features.hdf5` 존재 여부만 확인하는지 확인하지 못해 **작은 더미 파일**을 만들어 둡니다.
- 시간이 오래 걸립니다(README에 예상 시간 없음). 세션이 끊기면 `preprocessed_data/` 캐시를 재사용하므로 같은 셀을 다시 실행하면 이어집니다.

In [ ]:
# 10. 노트북 인자 패치 + 실행 도우미
import json, re, subprocess
def patch_notebook(src_nb, dst_nb, repl):
    nb = json.load(open(src_nb))
    hit = 0
    for c in nb["cells"]:
        if c["cell_type"] != "code":
            continue
        s = "".join(c["source"])
        if "jupyter_args" in s and "f\"" in s:
            s2 = re.sub(r'jupyter_args\s*=\s*f"[^"]*"', lambda m: "jupyter_args = " + repr(repl) , s, flags=re.S)
            if s2 != s:
                c["source"] = s2.splitlines(keepends=True)
                hit += 1
    json.dump(nb, open(dst_nb, "w"))
    print("patched cells:", hit)
    return hit
def run_nb(path, out):
    cmd = ["/content/mei-env/bin/jupyter", "nbconvert", "--to", "notebook", "--execute", path, "--output", out,
           "--ExecutePreprocessor.kernel_name=mei-env", "--ExecutePreprocessor.timeout=-1"]
    print(" ".join(cmd))
    r = subprocess.run(cmd, cwd=SRC)
    return r.returncode

In [ ]:
# 11. 학습 실행
import h5py
git_path = f"{DATA}/git_image_features.hdf5"
if not USE_PROMPT_RECON and not os.path.exists(git_path):
    with h5py.File(git_path, "w") as f:
        f.create_dataset("features", data=np.zeros((1,), np.float32))   # 더미: 프롬프트 구성요소를 끈 경우 건너뛰기 위함
flags = ["--dual_guidance" if USE_DUAL_GUIDANCE else "--no-dual_guidance",
         "--prompt_recon" if USE_PROMPT_RECON else "--no-prompt_recon",
         "--blurry_recon" if USE_BLURRY else "--no-blurry_recon"]
args = (f"--data_path=../dataset/ --cache_dir=../cache/ --model_name={MODEL_NAME} --batch_size=64 --no-multi_subject "
        f"--subj={SUBJ} --num_sessions={NUM_SESSIONS} --caption_type {CAPTION_TYPE} " + " ".join(flags))
patch_notebook(f"{SRC}/Train.ipynb", f"{SRC}/Train_colab.ipynb", args)
rc = run_nb("Train_colab.ipynb", "Train_colab_out.ipynb")
print("return code", rc)

In [ ]:
# 12. 학습 결과 점검
tl = f"{ROOT}/train_logs/{MODEL_NAME}"
print(os.listdir(tl) if os.path.exists(tl) else "⚠️ train_logs가 없음: 위 실행 로그를 확인하세요")

## D. 생성 (재구성)
`src/recon_inference_mi.ipynb`를 인자만 바꿔서 실행합니다. 사전에 NSD-Imagery 베타를 `preprocessed_data/subject{N}/nsd_imagery.pt`로 만들어야 합니다(저장소의 `utils.create_whole_region_imagery_*`).

In [ ]:
# 13. nsd_imagery.pt 만들기: 함수 시그니처를 먼저 확인
check = f'''
import sys, inspect
sys.path.insert(0, "{SRC}")
import utils
for n in ("create_whole_region_imagery_unnormalized", "create_whole_region_imagery_normalized"):
    f = getattr(utils, n, None)
    print(n, inspect.signature(f) if f else "없음")
'''
open("/content/_sig.py", "w").write(check)
!cd {SRC} && /content/mei-env/bin/python /content/_sig.py

In [ ]:
# 14. 위 시그니처에 맞게 호출(키워드 이름이 다르면 고치세요). GLMdenoise=True, mask=True가 기본 파일 nsd_imagery.pt를 만든다.
code_ = f'''
import sys
sys.path.insert(0, "{SRC}")
import utils
kw = dict(subject={SUBJ}, mask=True, GLMdenoise=True, data_path="{DATA}")
utils.create_whole_region_imagery_unnormalized(**kw)
utils.create_whole_region_imagery_normalized(**kw)
'''
open("/content/_mk.py", "w").write(code_)
!cd {SRC} && /content/mei-env/bin/python /content/_mk.py
print(os.listdir(f"{DATA}/preprocessed_data/subject{SUBJ}"))

In [ ]:
# 15. 생성 실행 (모드마다)
for MODE in MODES:
    flags = ["--dual_guidance" if USE_DUAL_GUIDANCE else "--no-dual_guidance",
             "--prompt_recon" if USE_PROMPT_RECON else "--no-prompt_recon",
             "--blurry_recon" if USE_BLURRY else "--no-blurry_recon"]
    args = (f"--data_path=../dataset --cache_dir=../cache --model_name={MODEL_NAME} --subj={SUBJ} --mode {MODE} "
            f"--gen_rep={GEN_REP} --num_images_per_sample={N_IMG} --num_trial_reps={NUM_TRIAL_REPS} " + " ".join(flags))
    patch_notebook(f"{SRC}/recon_inference_mi.ipynb", f"{SRC}/recon_{MODE}_colab.ipynb", args)
    print(MODE, "return code", run_nb(f"recon_{MODE}_colab.ipynb", f"recon_{MODE}_out.ipynb"))
print(os.listdir(f"{ROOT}/evals/{MODEL_NAME}") if os.path.exists(f"{ROOT}/evals/{MODEL_NAME}") else "⚠️ evals가 없음")

## E. 시각화와 재현 정도
정답(목표 자극), 저수준(blurry) 재구성, 최종 재구성을 나란히 보고, CLIP ViT-L/14 임베딩으로 **18개 자극 중 정답 찾기** 정확도를 계산합니다(우연 1/18). 저장 형식은 확인하지 못해 형태를 출력하고 일반적인 (N,3,H,W) 텐서를 가정합니다.

In [ ]:
# 16. 결과 불러오기와 그림
import torch, matplotlib.pyplot as plt, numpy as np
MODE = MODES[0]
EV = f"{ROOT}/evals/{MODEL_NAME}"
recons = torch.load(f"{EV}/{MODEL_NAME}_all_recons_{MODE}.pt", map_location="cpu")
blurry = torch.load(f"{EV}/{MODEL_NAME}_all_blurryrecons_{MODE}.pt", map_location="cpu") if os.path.exists(f"{EV}/{MODEL_NAME}_all_blurryrecons_{MODE}.pt") else None
gt = torch.load(f"{DATA}/nsddata_stimuli/stimuli/imagery_stimuli_18.pt", map_location="cpu")
print("recons", getattr(recons, "shape", type(recons)), "blurry", getattr(blurry, "shape", None), "gt", gt.shape)
def to_img(t):
    t = t.float()
    if t.min() < 0: t = (t + 1) / 2
    return t.clamp(0, 1).permute(1, 2, 0).numpy()
n = min(len(recons), len(gt))
rows = 3 if blurry is not None else 2
fig, axs = plt.subplots(rows, n, figsize=(1.6 * n, 1.8 * rows))
for i in range(n):
    axs[0, i].imshow(to_img(gt[i])); axs[0, i].axis("off")
    axs[1, i].imshow(to_img(recons[i])); axs[1, i].axis("off")
    if blurry is not None:
        axs[2, i].imshow(to_img(blurry[i])); axs[2, i].axis("off")
axs[0, 0].set_title("정답", fontsize=8, loc="left"); axs[1, 0].set_title("재구성", fontsize=8, loc="left")
plt.tight_layout(); plt.savefig(f"{ROOT}/recon_grid_{MODE}.png", dpi=150); plt.show()

In [ ]:
# 17. CLIP 식별 정확도(18개 중 정답 찾기, 우연 1/18)
%pip -q install transformers
from transformers import CLIPModel, CLIPProcessor
dev = "cuda"
cm = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(dev).eval()
pr = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
@torch.no_grad()
def emb(ts):
    ims = [(to_img(t) * 255).astype(np.uint8) for t in ts]
    f = cm.get_image_features(**pr(images=ims, return_tensors="pt").to(dev))
    f = f.pooler_output if hasattr(f, "pooler_output") else f
    return torch.nn.functional.normalize(f, dim=-1).cpu()
E_gt, E_rc = emb(gt[:n]), emb(recons[:n])
S = E_rc @ E_gt.T
top1 = (S.argmax(1) == torch.arange(n)).float().mean().item()
top1b = (S.argmax(0) == torch.arange(n)).float().mean().item()
print(f"재구성 → 정답 식별 top-1 {top1:.3f}, 정답 → 재구성 {top1b:.3f}, 우연 {1/n:.3f}")
plt.figure(figsize=(4, 4)); plt.imshow(S.numpy(), cmap="viridis"); plt.colorbar(); plt.xlabel("정답"); plt.ylabel("재구성"); plt.savefig(f"{ROOT}/clip_sim_{MODE}.png", dpi=150); plt.show()

In [ ]:
# 18. 결과 Drive 백업(선택)
import shutil
for p in (f"{ROOT}/train_logs/{MODEL_NAME}", f"{ROOT}/evals/{MODEL_NAME}"):
    if os.path.exists(p):
        shutil.copytree(p, f"{DRIVE_DIR}/{os.path.basename(os.path.dirname(p))}/{MODEL_NAME}", dirs_exist_ok=True)
for f in (f"{ROOT}/recon_grid_{MODE}.png", f"{ROOT}/clip_sim_{MODE}.png"):
    if os.path.exists(f): shutil.copy(f, DRIVE_DIR)
print("saved to", DRIVE_DIR)